In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import numpy as np
from scipy import stats

import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd


#方差分析检验三组数据的显著性，ANOVA
def anova(data1,data2,data3):
    # 进行单因素方差分析
    f_statistic, p_value = stats.f_oneway(data1,data2,data3)

    print("F统计量:", f_statistic)
    print("p值:", p_value)

    # 检验结果
    alpha = 0.05
    if p_value < alpha:
        print("拒绝零假设，至少有一组的均值与其他组显著不同。")
    else:
        print("无法拒绝零假设，各组的均值没有显著差异。")
    
    return f_statistic, p_value


def tukey_HSD(data1, data2, data3):
    # 将数据整理为DataFrame
    data = {
        'score': np.concatenate([data1, data2, data3]),
        'group': ['group1'] * len(data1) + ['group2'] * len(data2) + ['group3'] * len(data3)
    }
    df = pd.DataFrame(data)

    # 进行方差分析
    model = ols('score ~ group', data=df).fit()
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(anova_table)

    # 进行Tukey HSD事后比较
    tukey = pairwise_tukeyhsd(endog=df['score'], groups=df['group'], alpha=0.05)
    print(tukey)
    
    return anova_table, tukey

def draw_fig(fs, data, save_path):
    # 将数据转换为 DataFrame
    df = pd.DataFrame(data)

    # 设置 Seaborn 样式
    sns.set(style="whitegrid")

    # 绘制箱形图
    plt.figure(figsize=(10, 6))
    # sns.boxplot(data=df, palette='viridis')
    sns.boxplot(data=df, palette=['#1f77b4', '#ff7f0e', '#2ca02c'])
    # 添加标题和标签
    plt.title('Box Plot of Sample Distribution')
    plt.xlabel('three types of summary score')
    plt.ylabel('%s score'%(fs))
#     plt.savefig(save_path+fs+'.jpg', dpi=None, bbox_inches='tight', pad_inches=0.1)
    # 显示图形
    plt.tight_layout()
    plt.show()

In [ ]:

from data_loader_saver import load_data
import os 
from tqdm import tqdm

save_path = '/root/autodl-fs/zyq/DeFacto/data/fig/'
text_list=['abstract','candidate','humman_summary']
graph_sims = {}

file_list = ['test','val','train']
for file in file_list:
    factgraph_output_file_path = '/root/autodl-fs/zyq/DeFacto/data/graph_similarity_result/%s/'%(file)
    filenames = os.listdir(factgraph_output_file_path)
    for filename in tqdm(filenames):
        if filename.endswith('.json'):
            file_path = os.path.join(factgraph_output_file_path, filename)
            graph_data = load_data(file_path)

            intrinsic_error = graph_data['intrinsic_error']
            extrinsic_error = graph_data['extrinsic_error']
            if intrinsic_error == True and extrinsic_error == False:

                key1 = 'article'
                # print(graph_data['doc_id'])
                for key2 in text_list:
                    key = '%s_%s'%(key1,key2)
                    if '%s_%s'%(key1,key2) not in graph_sims.keys():
                        graph_sims[key] = []

                    has_nan = np.isnan(graph_data['graph_sim'][key])
                    if has_nan:
                        print("数组中有NaN值:", graph_data['doc_id'],len(graph_sims[key]))
                        graph_sims[key].append(0)
                    else:
                        graph_sims[key].append(graph_data['graph_sim'][key])

In [ ]:
for k, v in graph_sims.items():
    graph_sims[k] = np.array(v)
    
#     检查是否有NaN和None值  
#     has_nan = np.isnan(v).any()
#     print("数组中是否有NaN值:", has_nan,k)
#     if has_nan:
#         nan_indices = np.where(np.isnan(v))
#         print("NaN 值的位置:", nan_indices,v[nan_indices])
#     has_none = None in graph_sims[k]
#     print("数组中是否有None值:", has_none,k)
data1 = graph_sims['article_abstract']
data2 = graph_sims['article_candidate']
data3 = graph_sims['article_humman_summary']   
draw_fig('FactGraph', graph_sims, save_path)
f, p = anova(data1, data2, data3)
anova_table, tukey = tukey_HSD(data1, data2, data3)

In [ ]:

from data_loader_saver import load_data
import os 
from tqdm import tqdm

save_path = '/root/autodl-fs/zyq/DeFacto/data/fig/'
text_list=['abstract','candidate','humman_summary']
graph_sims_ext = {}

file_list = ['test','val','train']
for file in file_list:
    factgraph_output_file_path = '/root/autodl-fs/zyq/DeFacto/data/graph_similarity_result/%s/'%(file)
    filenames = os.listdir(factgraph_output_file_path)
    for filename in tqdm(filenames):
        if filename.endswith('.json'):
            file_path = os.path.join(factgraph_output_file_path, filename)
            graph_data = load_data(file_path)

            intrinsic_error = graph_data['intrinsic_error']
            extrinsic_error = graph_data['extrinsic_error']
            if intrinsic_error == False and extrinsic_error == True:

                key1 = 'article'
                # print(graph_data['doc_id'])
                for key2 in text_list:
                    key = '%s_%s'%(key1,key2)
                    if '%s_%s'%(key1,key2) not in graph_sims_ext.keys():
                        graph_sims_ext[key] = []

                    has_nan = np.isnan(graph_data['graph_sim'][key])
                    if has_nan:
                        print("数组中有NaN值:", graph_data['doc_id'],len(graph_sims_ext[key]))
                        graph_sims_ext[key].append(0)
                    else:
                        graph_sims_ext[key].append(graph_data['graph_sim'][key])

In [ ]:
for k, v in graph_sims_ext.items():
    graph_sims_ext[k] = np.array(v)
    
#     检查是否有NaN和None值  
#     has_nan = np.isnan(v).any()
#     print("数组中是否有NaN值:", has_nan,k)
#     if has_nan:
#         nan_indices = np.where(np.isnan(v))
#         print("NaN 值的位置:", nan_indices,v[nan_indices])
#     has_none = None in graph_sims[k]
#     print("数组中是否有None值:", has_none,k)
data1 = graph_sims_ext['article_abstract']
data2 = graph_sims_ext['article_candidate']
data3 = graph_sims_ext['article_humman_summary']   
draw_fig('FactGraph', graph_sims, save_path)
f, p = anova(data1, data2, data3)
anova_table, tukey = tukey_HSD(data1, data2, data3)

In [ ]:
from data_loader_saver import load_data
import json
from tqdm import tqdm

factgraph_output_file_path = '/root/autodl-fs/zyq/DeFacto/data/all_data_evaluate.json'
save_path = '/root/autodl-fs/zyq/DeFacto/data/fig/'
text_list=['abstract','candidate','humman_summary']
summary_type_list = ['abstract_score','candidate_score','humman_summary_score']
fact_score_list=['cloze','dae_doc','factcc','summacconv','quals','feqa','q2']
with open(factgraph_output_file_path, 'r', encoding='utf-8') as f1:
    graph_data = json.load(f1)
    
fact_summary_score={}

for item in graph_data:
    intrinsic_error = item['intrinsic_error']
    extrinsic_error = item['extrinsic_error']
    if intrinsic_error ==False and extrinsic_error == True:
        for fs in fact_score_list:
            if fs not in fact_summary_score.keys():
                fact_summary_score[fs]={}
            for st in summary_type_list:
                factscore = item[st][fs]
                if st not in fact_summary_score[fs].keys():

                    fact_summary_score[fs][st]=[]
                fact_summary_score[fs][st].append(factscore)

In [ ]:
for fs in fact_score_list:
    all_type_summary_score = fact_summary_score[fs]
    
    for k, v in all_type_summary_score.items():
        all_type_summary_score[k] = np.array(v)
    data1 = all_type_summary_score['abstract_score']
    data2 = all_type_summary_score['candidate_score']
    data3 = all_type_summary_score['humman_summary_score']
    print(fs)
    f, p = anova(data1, data2, data3)
    draw_fig(fs, all_type_summary_score, save_path)
    anova_table, tukey = tukey_HSD(data1, data2, data3)

In [ ]:
#统计人类摘要高于模型摘要的条数。

from data_loader_saver import load_data
import os 
from tqdm import tqdm

save_path = '/root/autodl-fs/zyq/DeFacto/data/fig/'
text_list=['abstract','candidate','humman_summary']
graph_sims_ext = {}

text_list=['abstract','candidate','humman_summary']
hum_larger_than_cand = 0
count = 0

file_list = ['test','val','train']
for file in file_list:
    factgraph_output_file_path = '/root/autodl-fs/zyq/DeFacto/data/graph_similarity_result/%s/'%(file)
    filenames = os.listdir(factgraph_output_file_path)
    for filename in tqdm(filenames):
        if filename.endswith('.json'):
            file_path = os.path.join(factgraph_output_file_path, filename)
            graph_data = load_data(file_path)
            intrinsic_error = graph_data['intrinsic_error']
            extrinsic_error = graph_data['extrinsic_error']
            graph_sim = graph_data['graph_sim']
            if intrinsic_error == True and extrinsic_error == False:
                if graph_sim['article_humman_summary'] > graph_sim['article_candidate']:
                        hum_larger_than_cand += 1
                count += 1
print(hum_larger_than_cand)#249   55
print(count)#339  98

In [ ]:
from data_loader_saver import load_data
import json
from tqdm import tqdm

factgraph_output_file_path = '/root/autodl-fs/zyq/DeFacto/data/all_data_evaluate.json'

text_list=['abstract','candidate','humman_summary']
summary_type_list = ['abstract_score','candidate_score','humman_summary_score']
fact_score_list=['cloze','dae_doc','factcc','summacconv','quals','feqa','q2']
with open(factgraph_output_file_path, 'r', encoding='utf-8') as f1:
    graph_data = json.load(f1)
fact_summary_score={}
for item in graph_data:
    intrinsic_error = item['intrinsic_error']
    extrinsic_error = item['extrinsic_error']
    if intrinsic_error ==False and extrinsic_error == True:
        for fs in fact_score_list:
            if fs not in fact_summary_score.keys():
                fact_summary_score[fs]={}
            for st in summary_type_list:
                factscore = item[st][fs]
                if st not in fact_summary_score[fs].keys():

                    fact_summary_score[fs][st]=[]
                fact_summary_score[fs][st].append(factscore)

In [ ]:
import numpy as np

for fs in fact_score_list:
    all_type_summary_score = fact_summary_score[fs]
    print(fs)
    hum_larger_than_cand = 0
    
    for k, v in all_type_summary_score.items():
        all_type_summary_score[k] = np.array(v)
    # data1 = all_type_summary_score['abstract_score']
    data2 = all_type_summary_score['candidate_score']
    data3 = all_type_summary_score['humman_summary_score']
    hum_larger_than_cand = np.sum(data3 > data2)

    print(hum_larger_than_cand)
    print(len(data2))


cloze
157
339
dae_doc
181
339
factcc
41
339
summacconv
196
339
quals
230
339
feqa
210
339
q2
203
339

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 数据
data = {
    'metric': ['factGraph', 'quals', 'feqa','q2','dae_doc','summacconv', 'cloze',  'factcc' ],
    'number': [249,230, 210, 203, 196, 181, 157, 41, ]
}

# 创建DataFrame
df = pd.DataFrame(data)

# 绘制柱状图
plt.figure(figsize=(10, 6))
plt.bar(df['metric'], df['number'], color='skyblue')
plt.title('Model Performance Comparison', fontsize=16)
plt.xlabel('fact matrics', fontsize=14)
plt.ylabel('number of right samples', fontsize=14)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# 显示图形
plt.tight_layout()
plt.show()
